
# <font color="green">Conditional select (add or multiply)</font>

## Problem

* Write a function `add_or_mul_long` __in assembly__, using a conditional select (`csel`) instead of a conditional branch.
* That is, translate the following C function into assembly:
```
long add_or_mul_long(long x, long y, long z) {
  if (x < y) { return y + z; } else { return y * z; }
}
```
* Compute both `y + z` and `y * z`, then select the result with `csel` according to the comparison of `x` and `y`.
* Fill in the skeleton `add_or_mul_long.s` (after `// ------- write your answer here -------`).
* The checker `check_add_or_mul_long.c` verifies your result. If you see `OK`s and no errors, you are done.

## Hints

* Superscalar processors decode instructions far ahead of the one currently executing. When they hit a branch, they *predict* its outcome; a misprediction forces the processor to roll back, which is costly.
* As such, compilers avoid branch instructions where profitable and use _conditional instructions_ instead.
* The *Observe* cells below contain `imax` (the maximum of two values). Compile it and notice that the compiler does **not** branch: it uses a `cmp` followed by `csel` ("conditional select"), which copies one of two source registers into the destination depending on the comparison.
* The same idea applies to this problem: compute `y + z` into one register and `y * z` into another, then use a single `csel` to keep the right one according to the comparison of `x` and `y` --- no branch at all. Computing both is profitable when both expressions are cheap.
* Related instructions you may see: `cset` (set a register to 0/1 from a condition), `cinc`, `csneg`, etc.



# 1. AI Tutor
## 1-1. Prepare
* Your personal AI tutor is provided for questions and feedback.
* Execute the following cell before you use it.

In [ ]:
import heytutor

## 1-2. Examples
* A general question
```
%%hey
What does the `ldr` instruction do in ARM64?
```

* A hint on this specific problem
```
%%hey problem_file=add_or_mul_long.md
Give me a hint on this problem.

{problem}
```

* Builtin variables usable in `%%hey` cells
  * `{file:FILENAME}` is the content of FILE
  * `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` the second last, etc.
  * `{problem}` is the content of the file you specify by `%%hey problem_file=foo.md`
  * `{answer}` is the content of the file you specify by `%%hey answer_file=foo.s`


# 2. Observe: compile example functions
* Before writing your own assembly, it helps to see what the compiler generates for related example functions.
* Running the first cell below writes `explore.c` (some small example functions related to this problem).
* The second cell compiles it with `gcc -O3 -S` and prints the generated assembly.
* Feel free to edit `explore.c` (change the code, add functions, change constants) and re-run the two cells to see how the assembly changes.

In [ ]:
%%writefile_ explore.c
/* if-without-branch: the compiler evaluates a value for each side and picks one
   with a conditional-select (csel) --- no branch at all. Observe `cmp` followed
   by `csel`. (Here the two candidates are simply `a` and `b`.) */
long imax(long a, long b) {
  return a > b ? a : b;
}

In [ ]:
%%bash_
gcc -O3 -S explore.c
cat explore.s


# 3. Your Answer (assembly)
* Running the cell below writes the skeleton assembly file `add_or_mul_long.s`.
* Fill in your instructions after the line `// ------- write your answer here -------`, then run the cell again to save it.

In [ ]:
%%writefile_ add_or_mul_long.s
	.arch armv8-a
	.file	"add_or_mul_long.c"
	.text
	.align	2
	.p2align 4,,11
	.global	add_or_mul_long
	.type	add_or_mul_long, %function
add_or_mul_long:
.LFB0:
	.cfi_startproc
	// ------- write your answer here -------
	.cfi_endproc
.LFE0:
	.size	add_or_mul_long, .-add_or_mul_long
	.ident	"GCC: (Ubuntu 13.3.0-6ubuntu2~24.04) 13.3.0"
	.section	.note.GNU-stack,"",@progbits


# 4. Checker
* The following C program calls your `add_or_mul_long` function and checks the result against a reference computed in C.

In [ ]:
%%writefile_ check_add_or_mul_long.c
#include <assert.h>
#include <stdio.h>
#include <stdlib.h>
long add_or_mul_long(long x, long y, long z);
int main(int argc, char ** argv) {
  assert(argc == 4);
  long x = atol(argv[1]);
  long y = atol(argv[2]);
  long z = atol(argv[3]);
  long r = add_or_mul_long(x, y, z);
  long rc = (x < y) ? (y + z) : (y * z);
  if (r == rc) { printf("OK %ld %ld\n", r, rc); return 0; }
  else { printf("NG %ld %ld\n", r, rc); return 1; }
}


# 5. Compile
* Compile your assembly together with the checker.
* If you get an error, fix `add_or_mul_long.s` above and recompile.

In [ ]:
%%bash_
gcc -o check_add_or_mul_long -O3 check_add_or_mul_long.c add_or_mul_long.s -lm


# 6. Run
* If you see `OK`s and no errors, you are done.

In [ ]:
%%bash_
./check_add_or_mul_long 1 2 3
./check_add_or_mul_long 5 2 3
./check_add_or_mul_long 2 2 3


# 7. If things do not go well
* If your program compiles but does not produce the correct answer, run it within a debugger (gdb).
* Compile with `-O0 -g` first:
```
gcc -o check_add_or_mul_long -O0 -g check_add_or_mul_long.c add_or_mul_long.s -lm
```
* Then, in a terminal (SSH or Jupyter terminal):
```
gdb check_add_or_mul_long
(gdb) break add_or_mul_long
(gdb) run ...        # give the same arguments as above
```
* Step through one instruction at a time with `step`, and inspect registers with `print $x0` or `info registers`.

# 8. Ask Questions or Get Feedback
* You are encouraged to ask for feedback once you think you are done, to know if there is a better answer.

In [ ]:
%%hey problem_file=add_or_mul_long.md answer_file=add_or_mul_long.s

Problem:
{problem}

My Answer:
{answer}

Give me a feedback to my answer.